## 🎯 Learning Objectives
* Understand the limitations of single-index RAG systems for complex and diverse data.
* Grasp the core concepts of query routing and multi-index strategies in RAG.
* Learn how to implement a multi-index RAG system using LlamaIndex's RouterQueryEngine.
* Analyze the performance trade-offs and identify appropriate use cases for advanced RAG patterns.


## Query Routing and Multi-Index Strategies: Navigating Complex Knowledge Bases

In the rapidly evolving landscape of Retrieval Augmented Generation (RAG), moving beyond a single, monolithic knowledge base is crucial for building robust, scalable, and accurate production systems. As enterprises accumulate vast amounts of diverse data—from financial reports and product documentation to HR policies and customer support tickets—a single RAG index often becomes a bottleneck. It struggles with relevance, efficiency, and the sheer complexity of distinguishing between different domains of information.

### The Challenge: Information Overload and Irrelevance
Imagine a vast library where all books, regardless of their subject (fiction, science, history, finance), are shelved together in a single, undifferentiated section. When you ask for 'information on quantum computing,' the librarian has to sift through *every single book* to find relevant ones. This is akin to a single-index RAG system querying a massive, undifferentiated vector store. It's inefficient, prone to retrieving irrelevant information, and computationally expensive.

### The Solution: Specialized Sections and a Smart Librarian
Now, imagine that same library is organized into specialized sections: a 'Science & Technology' wing, a 'Business & Finance' section, a 'Human Resources' department, and so on. Each section has its own dedicated catalog (index). When you ask for 'quantum computing,' a smart librarian (the **Query Router**) first determines *which section* is most likely to hold the answer. They then direct your query *only* to that specific section's catalog. This is the essence of **Query Routing** and **Multi-Index Strategies**.

#### What is Query Routing?
Query routing is the process of intelligently directing an incoming user query to the most appropriate underlying data source or RAG index. Instead of querying all available data, the router analyzes the query's intent and content to select the best-fit knowledge base. This decision can be based on keywords, semantic similarity, explicit metadata, or even a sophisticated LLM-driven classification.

**Benefits of Query Routing:**
1.  **Improved Relevance:** Queries are directed to specialized, highly relevant data, reducing noise and improving answer quality.
2.  **Enhanced Efficiency:** Only a subset of indices is queried, significantly reducing retrieval time and computational costs.
3.  **Scalability:** Easily add new data sources and indices without re-indexing the entire knowledge base.
4.  **Modularity:** Different data domains can be managed, updated, and optimized independently.
5.  **Cost Optimization:** Lower API calls to vector databases or LLMs by focusing retrieval.

#### What are Multi-Index Strategies?
Multi-index strategies involve creating and managing multiple distinct RAG indices, each tailored to a specific domain, data type, or purpose. These indices can be of various types:

*   **Vector Store Indices:** For semantic search over unstructured text (e.g., documents, articles).
*   **Keyword Indices:** For exact or fuzzy keyword matching (e.g., product IDs, specific terms).
*   **Knowledge Graph Indices:** For structured relationships and entities (e.g., organizational charts, product dependencies).
*   **Document Summary Indices:** For high-level overviews of large documents.

By combining these different index types and routing queries between them, we can build highly sophisticated and performant RAG systems capable of handling a wide array of complex information needs. LlamaIndex, as a leading framework for RAG, provides powerful abstractions like the `RouterQueryEngine` to seamlessly implement these advanced patterns.

In the following example, we'll simulate a scenario with two distinct knowledge bases—one for 'Financial Reports' and another for 'Product Documentation'—and build a router to direct queries to the appropriate source.


In [ ]:
# Ensure you have the necessary libraries installed:
# pip install llama-index llama-index-llms-openai llama-index-embeddings-openai

import os
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# --- 1. Configuration and API Keys (2026 Ready: Use environment variables) ---
# It's 2026, so we're using secure environment variables for API keys.
# Make sure to set your OPENAI_API_KEY environment variable.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

if "OPENAI_API_KEY" not in os.environ:
    raise ValueError("OPENAI_API_KEY environment variable not set. Please set it to run this example.")

# Configure LlamaIndex global settings for LLM and Embedding models
# Using modern, performant models available in 2026
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.1)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

print("LlamaIndex settings configured with gpt-4o-mini and text-embedding-3-small.")

# --- 2. Simulate Diverse Data Sources ---
# In a real-world scenario, these would be loaded from files, databases, etc.

financial_docs_text = [
    "Annual Report 2025: Revenue grew by 15% to $500M. Net profit reached $80M. Key initiatives included AI integration and sustainable energy investments.",
    "Q3 2025 Earnings Call Transcript: Discussed market expansion into APAC and new partnerships. Forecasted 10% growth for Q4.",
    "Investment Strategy 2026: Focus on generative AI startups and quantum computing research. Allocated $100M for R&D."
]

product_docs_text = [
    "Product X User Manual: Features include real-time data analytics, secure cloud integration, and a modular API for custom extensions.",
    "Product Y Technical Specifications: Built on a microservices architecture using Kubernetes. Supports up to 1M concurrent users. Requires Python 3.10+.",
    "Product Roadmap 2026: Upcoming features include multi-modal input support, enhanced security protocols, and a new mobile application for Product X."
]

# Create LlamaIndex Document objects
financial_documents = [Document(text=t, metadata={"source": "financial"}) for t in financial_docs_text]
product_documents = [Document(text=t, metadata={"source": "product"}) for t in product_docs_text]

print(f"Created {len(financial_documents)} financial documents and {len(product_documents)} product documents.")

# --- 3. Create Separate Indices for Each Data Source ---
print("Creating VectorStoreIndex for financial documents...")
financial_index = VectorStoreIndex.from_documents(financial_documents)
financial_query_engine = financial_index.as_query_engine(similarity_top_k=2)

print("Creating VectorStoreIndex for product documents...")
product_index = VectorStoreIndex.from_documents(product_documents)
product_query_engine = product_index.as_query_engine(similarity_top_k=2)

print("Indices created.")

# --- 4. Define QueryEngineTools for the Router ---
# Each tool wraps a query engine and provides a description for the router LLM.
financial_tool = QueryEngineTool(
    query_engine=financial_query_engine,
    metadata={
        "name": "financial_reports_engine",
        "description": "Provides information about company financial performance, earnings, investments, and market strategies."
    }
)

product_tool = QueryEngineTool(
    query_engine=product_query_engine,
    metadata={
        "name": "product_documentation_engine",
        "description": "Provides technical details, user manuals, features, and roadmap information for company products."
    }
)

print("QueryEngineTools defined.")

# --- 5. Build the RouterQueryEngine ---
# The LLMSingleSelector uses an LLM to decide which tool to use.
router_query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[financial_tool, product_tool]
)

print("RouterQueryEngine initialized.")

# --- 6. Test the Router with Various Queries ---
print("\n--- Testing Queries ---")

queries = [
    "What was the company's revenue in 2025?",
    "Tell me about the features of Product X.",
    "What are the key investment areas for 2026?",
    "Which programming language is required for Product Y?",
    "What is the forecasted growth for Q4?",
    "What new features are planned for Product X in 2026?"
]

for i, query in enumerate(queries):
    print(f"\nQuery {i+1}: {query}")
    response = router_query_engine.query(query)
    print(f"Router selected tool: {response.metadata['selector_result'].selections[0].tool_name}")
    print(f"Answer: {response.response}")
    print("--------------------------------------------------")

print("\nDemonstration complete. Observe how the router intelligently directs queries.")


### Interpreting the Code Output and Performance Trade-offs

The code demonstrates how LlamaIndex's `RouterQueryEngine` effectively directs queries to specialized indices. For each query, you'll observe:

1.  **Query Routing Decision:** The output explicitly states which `tool_name` (e.g., `financial_reports_engine` or `product_documentation_engine`) the router selected. This confirms that the LLM-powered selector successfully understood the intent of the query and matched it to the most relevant data source based on the `description` provided in the `QueryEngineTool`.
2.  **Relevant Answer:** The answer provided by the RAG system is derived *only* from the selected index, showcasing improved relevance and reduced noise compared to querying a combined, undifferentiated index.

#### Performance Trade-offs and Considerations:

**Advantages:**

*   **Accuracy and Relevance:** By segmenting knowledge, the RAG system can provide more precise and contextually relevant answers, as the LLM is not distracted by irrelevant information from other domains.
*   **Efficiency and Speed:** Only a subset of the total data (the selected index) is searched, leading to faster retrieval times and lower computational load on vector databases and embedding models.
*   **Scalability:** New data domains can be added as separate indices without requiring a full re-indexing of the entire knowledge base. This is critical for large-scale enterprise RAG systems.
*   **Maintainability:** Each index can be optimized, updated, and managed independently, simplifying data governance and lifecycle management.
*   **Cost Reduction:** Fewer retrieval operations across massive datasets can lead to significant cost savings, especially with cloud-based vector stores and LLM API calls.

**Disadvantages:**

*   **Increased Complexity:** Setting up and managing multiple indices and a router adds an extra layer of architectural complexity compared to a single-index system. This includes defining clear boundaries for each index and crafting effective tool descriptions.
*   **Routing Overhead:** The router itself (often an LLM) incurs a small latency and cost overhead to make the routing decision. While usually negligible compared to retrieval, it's a factor.
*   **Potential for Routing Errors:** If the router's descriptions are ambiguous or the query is highly nuanced and crosses domain boundaries, the router might misdirect the query, leading to incorrect or incomplete answers. Robust testing and iterative refinement of tool descriptions are essential.
*   **Cross-Domain Queries:** For queries that genuinely require information from *multiple* domains (e.g., "What is the financial impact of the new features in Product X?"), a simple single-selector router might struggle. More advanced routing strategies (e.g., multi-selector, agentic routing) or hierarchical RAG might be needed for such cases.

#### Typical Use Cases:

*   **Enterprise Knowledge Bases:** Companies with distinct departments (HR, Legal, Finance, Engineering, Sales) each having their own documentation.
*   **Multi-Product Support:** Organizations supporting multiple products, each with its own extensive documentation.
*   **Regulatory Compliance:** Separating general company policies from specific regulatory documents that require different retrieval and security protocols.
*   **News and Media Aggregation:** Categorizing articles by topic (politics, sports, technology) and routing queries accordingly.
*   **Customer Service Bots:** Directing customer queries to product-specific FAQs, troubleshooting guides, or billing information.

By carefully designing your multi-index strategy and refining your query router, you can build highly effective and efficient RAG systems capable of navigating even the most complex information landscapes.


### Resources

*   **LlamaIndex Documentation - Query Routing:** [https://docs.llamaindex.ai/en/stable/module_guides/querying/router/root.html](https://docs.llamaindex.ai/en/stable/module_guides/querying/router/root.html)
*   **LlamaIndex Documentation - Query Engine Tools:** [https://docs.llamaindex.ai/en/stable/module_guides/querying/query_engine/root.html#query-engine-tools](https://docs.llamaindex.ai/en/stable/module_guides/querying/query_engine/root.html#query-engine-tools)
*   **LlamaIndex Documentation - Selectors:** [https://docs.llamaindex.ai/en/stable/module_guides/querying/router/selector.html](https://docs.llamaindex.ai/en/stable/module_guides/querying/router/selector.html)
*   **OpenAI API Documentation (for LLMs and Embeddings):** [https://platform.openai.com/docs/](https://platform.openai.com/docs/)
*   **Hugging Face Transformers (for alternative LLMs/Embeddings):** [https://huggingface.co/docs/transformers/index](https://huggingface.co/docs/transformers/index)
*   **Google AI Studio (for Gemini models):** [https://ai.google.dev/](https://ai.google.dev/)
